# D47 — FULL 90 câu bằng **ReAct + Calculator tool** (Qwen3-4B)

Dạng D47: *cho phương trình bậc hai $z^2-2(m+a)z+m^2=0$ với tham số thực
$m$, đếm số giá trị $m$ để tồn tại nghiệm $z_0$ có $|z_0|=R$*. Bản chạy
đầy đủ của phương pháp đã kiểm chứng tay nhiều câu trong
`KLTN_D47_ReAct_Calculator_1cau.ipynb`, bao gồm 10 câu khó nhất (7/10 là
câu Gemini cũng sai).

Model **không tự tính tay bất kỳ phép nào** — mọi phép tính đều gọi tool
`Calculator` (sympy, chính xác tuyệt đối, **có bộ nhớ biến**, hỗ trợ
**ẩn số tự do** `{'m'}` để $m$ có thể tồn tại như ẩn chưa biết
xuyên suốt phần lớn quá trình) theo vòng lặp ReAct thật:
`Thought → Action → Action Input → Observation → …`

Prompt được **ép mở đầu bằng `<think>\nThought:`** (forced prefix) —
model không còn quyền tự chọn viết văn xuôi mở đầu trước khi vào định dạng
ReAct, đúng bản sửa lỗi cuối cùng đã kiểm chứng ổn định trên nhiều câu liên
tiếp trong bản 1 câu.

## Điểm khác bản 1 câu: vòng lặp ReAct chạy THEO LÔ

Chạy tuần tự 90 câu sẽ mất hàng giờ. Ở đây mỗi **vòng** gọi vLLM **một lần**
cho tất cả các câu đang hoạt động (vLLM tự batching), rồi chạy Calculator
riêng cho từng câu, rồi generate tiếp.

Mỗi câu có **bộ nhớ biến riêng** (`MayTinh()` riêng), **ngân sách token
riêng**, và tự thoát khỏi lô khi viết xong `Final Answer`.

Hạn mức tool gọi/câu (`35          # van an toan chong loop vo han (D47: toi da ~20 luot tinh 3 truong hop + 4 luot doi chieu phuong an)`) đủ dư cho khoảng 20 lượt tính
qua cả 3 trường hợp biện luận theo $\Delta'$ cộng tối đa 4 lượt so khớp
phương án.

## Prompt & backend giống hệt bản 1 câu

Cell 4 (prompt) và phần backend `MayTinh` ở Cell 5 được **trích nguyên văn**
từ notebook 1 câu bằng script `scratch/build_react_full_D47.py` — không gõ
lại, nên không có nguy cơ lệch giữa 2 bản.

## Trước khi chạy

Upload `plan_solve_prompts_D47.json` (90 câu, đã sinh sẵn từ
`Sinh_them_cau_hoi/So_phuc_day_du.csv`, đã loại câu gốc STT 56 nằm trong
few-shot) thành Kaggle Dataset (slug gợi ý `d47-full90`), gắn vào notebook,
bật GPU.

Kết quả: `/kaggle/working/d47_react_calculator_full90.csv`

In [ ]:
!pip install -q -U vllm
!pip uninstall -y -q torchcodec
import vllm; print('vLLM:', vllm.__version__)

In [ ]:
import os, re, json, time
import pandas as pd
import torch
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

DATA_PATH = '/kaggle/input/d47-full90/plan_solve_prompts_D47.json'  # SUA NEU KHAC
OUT_PATH  = '/kaggle/working/d47_react_calculator_full90.csv'

MODEL = 'Qwen/Qwen3-4B'
# Giu NGUYEN tham so sinh da kiem chung o ban 1 cau.
TEMPERATURE, TOP_P, TOP_K, SEED = 0.3, 0.95, 20, 42
PRESENCE_PENALTY = 1.2
MAX_MODEL_LEN, MAX_NEW_TOKENS = 14336, 9216

AN_SO_TU_DO = {'m'}

# Dung 2 GPU neu co: gap doi KV cache va nhanh hon.
N_GPU = torch.cuda.device_count()
TENSOR_PARALLEL = 2 if N_GPU >= 2 else 1
print(f'So GPU: {N_GPU} -> tensor_parallel_size={TENSOR_PARALLEL}')

os.environ['TOKENIZERS_PARALLELISM'] = 'false'

with open(DATA_PATH, encoding='utf-8') as f:
    records = json.load(f)
print('So cau:', len(records))

tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
llm = LLM(model=MODEL, tensor_parallel_size=TENSOR_PARALLEL, dtype='float16',
          max_model_len=MAX_MODEL_LEN, gpu_memory_utilization=0.90,
          trust_remote_code=True, enforce_eager=True, seed=SEED)
print('San sang.')

In [ ]:
# =====================================================================
# PROMPT (trich nguyen van tu KLTN_D47_ReAct_Calculator_1cau.ipynb)
# =====================================================================
# Doi sang dang khac => chi can viet lai [C] PART_HUONGGIAI va
# [D] PART_FEWSHOT, giu nguyen [A] PART_TOOL va [B] PART_KIENTHUC.

# [A] PART_TOOL - HUONG DAN DUNG TOOL (DUNG CHUNG CHO MOI DANG TOAN)
# =====================================================================
PART_TOOL = r'''You are an elite, algorithmic mathematical solver. Your primary directive is STRICT COMPLIANCE: you follow the given solution method exactly, you never improvise a different method, and you never guess.

**YOU HAVE A CALCULATOR. IT PERFORMS EVERY COMPUTATION; YOU PERFORM NONE.**

Your own mental arithmetic is unreliable, from the very first step of a solution to the very last, including steps that look "obvious" (plugging a number into a formula, comparing one value against another). Every number you use must come from the Calculator, and every `Observation` it returns is final: use it and move on. Never recompute it in your head to check it, never re-derive it in prose, never doubt it. The only thing ever worth reconsidering is whether the *expression you are about to send* is the right one; not the answer that comes back.

**HOW TO CALL THE CALCULATOR (ReAct format), one step at a time:**

```
Thought: <ONE short sentence naming what you compute next; no arithmetic, no checking>
Action: Calculator
Action Input: <ONE expression>
```

Then **STOP WRITING IMMEDIATELY.** Do not write `Observation:` yourself, do not guess the result; the system runs the real calculator and appends the real `Observation: <result>` for you to continue from. If you ever catch yourself about to type a number right after `Action Input:`, stop: that line is where you hand control to the Calculator. Likewise, never describe several upcoming steps in prose before executing them; planning ahead in words is exactly how you lose track of what has actually been computed and end up calling the Calculator with names that don't exist yet or in the wrong order. Repeat this `Thought → Action → Action Input → Observation` cycle as many times as the solution needs.

**THE CALCULATOR HAS MEMORY; USE IT.** This is the most important feature:
- Write `name = expression` to compute a value **and store it under that name**.
  Example: `Action Input: t = -7/2 + 3/4*I` → `Observation: t = -7/2 + 3*I/4`
- From then on you can use that name inside later expressions instead of copying long numbers.
  Example: `Action Input: mod_t = Abs(t)` then later `Action Input: k = -(11/6)/mod_t * t`
- **ALWAYS store every intermediate result in a variable and refer to it by name afterwards.** NEVER copy a long number from an earlier `Observation` back into a later `Action Input` by hand; copying is exactly where mistakes happen. Let the memory do it.

**SYNTAX RULES for `Action Input` (one expression per call):**
- Imaginary unit is capital `I`; never lowercase `i`.
- Square root is `sqrt(...)`; never the `√` symbol.
- Power is `**` (e.g. `x**2`), not `^`.
- Plain fractions are already exact: writing `7/2` gives exactly seven-halves, never a decimal. You do NOT need any special function for fractions.
- Modulus of a complex number: `Abs(x)`. Conjugate: `conjugate(x)`.
- To test whether two expressions are **exactly** equal, write `Eq(left, right)`. The `Observation` will be `True` or `False`, decided exactly; never approximately.
- Work in exact fractions and radicals wherever an exact closed form exists; never approximate anything yourself. Some equations genuinely have no simple closed form, such as a cubic, quartic, or higher-degree polynomial that does not factor into nice roots: for those, `nroots(<polynomial>)` gives the Calculator's own numeric roots, exactly like a real handheld calculator's equation-solve mode; this is still the Calculator computing, not you approximating. `nroots(...)` returns EVERY root of the polynomial, including non-real ones when the polynomial's real roots don't account for its full degree; before doing anything else with a root, check whether it is actually real with `Abs(im(root)) < 1e-9` (lowercase `im`; `Im` is not recognized and silently fails to evaluate) and discard it immediately if not. For a root confirmed real, use `re(root)`; not the raw value; in every later threshold or substitution, since even a numerically-real root can carry a residual non-zero imaginary part too small to matter but large enough to break a direct comparison. Any comparison built on such numeric roots (a threshold like `t > 0`, or a self-consistency check like `Abs(a - b) < 1e-6`) should use a small tolerance instead of exact `Eq()`; everything that does not depend on a numeric root; in particular the final count of solutions and matching it against the answer options; stays exact as usual.

**IF THE OBSERVATION STARTS WITH `ERROR`:** your syntax was wrong. Read the message, then write a new `Thought` and a corrected `Action Input`. Do not give up, and do not fall back to computing it yourself.

**WHEN YOU ARE DONE:** write your final line as
`Final Answer: \boxed{<Letter>}`
and stop. No further Action after that.
'''


# =====================================================================
# [B] PART_KIENTHUC - KIEN THUC NEN SO PHUC (DUNG CHUNG MOI DANG SO PHUC)
# =====================================================================
PART_KIENTHUC = r'''
**PART 1: FOUNDATIONAL KNOWLEDGE OF COMPLEX NUMBERS**

1. **Definition**: a complex number is $z = a + bi$ with $a$ the real part, $b$ the imaginary part, and $i^2 = -1$.
2. **Operations**:
   - $(a+bi) \pm (c+di) = (a \pm c) + (b \pm d)i$
   - $(a+bi)(c+di) = (ac - bd) + (ad + bc)i$
   - Multiplying by $i$ rotates by 90°: $i(a+bi) = -b + ai$.
3. **Conjugate**: $\overline{z} = a - bi$ when $z = a+bi$. Note $\overline{i} = -i$.
4. **Modulus**: $|z| = \sqrt{a^2+b^2}$. Note $|z| = |\overline{z}| = |-z|$, and $|i \cdot z| = |z|$.
5. **Triangle inequality**: $|z_1 + z_2| \ge |z_1| - |z_2|$.
   - **Equality condition**: equality holds if and only if $z_1$ and $z_2$ point in *opposite* directions, i.e. $z_1 = k z_2$ for a **real** number $k < 0$ (specifically $k = -|z_1|/|z_2|$).
6. **Triangle inequality, three terms**: $|z_1 + z_2 + z_3| \ge |z_1 + z_2| - |z_3| \ge |z_1| - |z_2| - |z_3|$.
   - **Equality condition**: equality holds if and only if $z_2$ and $z_3$ point in the *same* direction as each other, and both point *opposite* to $z_1$.
7. **Direction facts**: if $u$ points opposite to $v$, then $u = -\dfrac{|u|}{|v|} \cdot v$.
8. **Division**: to write $\dfrac{z_1}{z_2}$ in standard $a+bi$ form, multiply numerator and denominator by $\overline{z_2}$ (the conjugate of the denominator): $\dfrac{z_1}{z_2} = \dfrac{z_1\overline{z_2}}{z_2\overline{z_2}} = \dfrac{z_1\overline{z_2}}{|z_2|^2}$. Since $|z_2|^2$ is a positive real number, this removes $i$ from the denominator.
9. **Real/imaginary part notation**: for $z=a+bi$ ($a,b$ real), write $\mathrm{Re}(z)=a$ and $\mathrm{Im}(z)=b$.
10. **Pure imaginary number**: $z$ is pure imaginary if and only if $\mathrm{Re}(z)=0$ **and** $\mathrm{Im}(z)\ne0$ (the number $0$ itself is real, not pure imaginary, so $\mathrm{Im}(z)\ne0$ must be checked separately and never dropped).
11. **Modulus equation as an algebraic relation**: for $z=x+yi$ and a constant $z_0=x_0+y_0i$, the equation $|z-z_0|=R$ squares to $(x-x_0)^2+(y-y_0)^2=R^2$, which expands to a relation containing the cluster $x^2+y^2$ (plus linear terms in $x,y$ plus constants). Two such relations sharing the same $x^2+y^2$ cluster can be subtracted to eliminate it, leaving a linear equation in $x,y$.
12. **Modulus of a product**: $|u \cdot v| = |u| \cdot |v|$ for any complex numbers $u,v$ (this is why $|z^2-A|$ can be factored into $|z-w|\cdot|z+w|$ once $A=w^2$).
13. **Cauchy-Schwarz (B.C.S) inequality**: for real numbers $a,b,x,y$: $(ax+by)^2 \le (a^2+b^2)(x^2+y^2)$, with equality if and only if $(a,b)$ and $(x,y)$ are proportional. This turns an equation containing a mixed term like $cx+dy$ into a one-sided bound purely in terms of $x^2+y^2$.
14. **Square root of a complex number**: given $A=p+qi$, to find $w=c+di$ with $w^2=A$, expand $w^2=(c^2-d^2)+2cd\,i$ and match parts: $c^2-d^2=p$ and $2cd=q$. Squaring both and adding gives $(c^2+d^2)^2=p^2+q^2$, so $c^2+d^2=|A|$. Combined with $c^2-d^2=p$, this gives $c^2=\dfrac{|A|+p}{2}$ and $d^2=\dfrac{|A|-p}{2}$ (both always $\ge0$). Take $c=\sqrt{c^2}\ge0$; the sign of $d$ must match the sign of $q$ so that $2cd=q$ holds (if $q=0$, either sign works).
15. **Complex roots of a real-coefficient quadratic**: for $z^2+bz+c=0$ with $b,c$ real, write $\Delta'=(b/2)^2-c$. If $\Delta'<0$, the equation has no real root; its two roots are a complex-conjugate pair $z=-\dfrac{b}{2}\pm\sqrt{|\Delta'|}\,i$, where $|\Delta'|=-\Delta'$ since $\Delta'<0$ here.
16. **Conjugation as a geometric reflection**: the point representing $\overline{z}$ is the mirror image, across the real axis ($Ox$), of the point representing $z$. Consequently, if every vertex of a figure is replaced by its conjugate, the resulting figure is the mirror image of the original across $Ox$, so the two figures are congruent (in particular, they have equal area).
17. **Modulus of a difference as distance**: for $z_1=x_1+y_1i$ and $z_2=x_2+y_2i$, $|z_1-z_2|=\sqrt{(x_1-x_2)^2+(y_1-y_2)^2}$ is exactly the distance between the points representing $z_1$ and $z_2$. This is why an equation like $|z-z_1|+|z-z_2|=K$ translates directly into $MA+MB=K$, where $M,A,B$ are the points representing $z,z_1,z_2$.
18. **Triangle inequality for three points**: for any three points $A,B,M$ in the plane, $MA+MB\ge AB$, with equality if and only if $M$ lies on the segment $AB$ (between $A$ and $B$, inclusive).
'''


# =====================================================================
# [C] PART_HUONGGIAI - HUONG GIAI RIENG CUA DANG D47 (THAY KHI DOI DANG)
# =====================================================================
PART_HUONGGIAI = r'''
**PART 2: THE SOLUTION METHOD FOR THIS PROBLEM TYPE**

Problem shape: a quadratic equation $z^2-2(m+a)z+m^2=0$ with a real parameter $m$ ($a$ a given real constant), and a given modulus $R>0$. Count how many values of $m$ make some root $z_0$ of the equation satisfy $|z_0|=R$.

Follow this method exactly; do not invent a shorter route, and do not skip any step below.

1. **Read off $a$ and $R$.** Match the given equation against the standard form $z^2-2(m+a)z+m^2=0$: whatever is added to $m$ inside the parentheses is $a$ (it can be negative, e.g. $z^2-2(m-4)z+m^2=0$ means $a=-4$). $R$ is the given modulus on the right of $|z_0|=R$.
2. **Compute $\Delta'$.** For $z^2+bz+c=0$, $\Delta'=(b/2)^2-c$; here $b/2=-(m+a)$ and $c=m^2$, so $\Delta'=(m+a)^2-m^2$, a quadratic expression in $m$. Build it with the Calculator and keep $m$ symbolic; do not substitute a number for $m$ yet. Initialize `count = 0`; this running tally is the only place `count` is ever set directly.
3. **Case 1: $\Delta'=0$.** Solve $\Delta'=0$ for $m$ (this typically gives one value). For that $m$, the equation has a real double root $z_0=-b/2=a+m$. Check $|z_0|=R$ with `Eq(Abs(z0), R)`. If it passes, increment `count = count + 1`; if not, discard.
4. **Case 2: $\Delta'<0$.** By the complex-roots fact from PART 1, write the root formula: $z_0=(a+m)\pm\sqrt{|\Delta'|}\,i$. Since $\Delta'<0$ here, $|\Delta'|=-\Delta'$, so $z_0=(a+m)\pm\sqrt{-\Delta'}\,i$; this is the real part and imaginary part of $z_0$. Square the modulus term by term: $|z_0|^2=(a+m)^2+\bigl(\sqrt{-\Delta'}\bigr)^2=(a+m)^2+(-\Delta')=(a+m)^2-\Delta'$. Now expand and simplify by substituting $\Delta'=(a+m)^2-m^2$: $|z_0|^2=(a+m)^2-\bigl[(a+m)^2-m^2\bigr]=m^2$. This is a general fact, true for every equation of this exact family regardless of $a$: the squared modulus of the complex root always equals $m^2$, exactly the constant term of the equation. So $|z_0|=R \iff m^2=R^2 \iff m=R$ or $m=-R$. For each of these two candidate values, check whether it actually satisfies $\Delta'<0$ (substitute it into $\Delta'$ and test `< 0`, since this is a strict inequality on a specific number, not an exact equality). If it passes, increment `count = count + 1`; if not, discard.
5. **Case 3: $\Delta'>0$.** The equation has two distinct real roots, so $|z_0|=R \iff z_0=R$ or $z_0=-R$. Substitute each of these two concrete numbers for $z$ directly into the ORIGINAL equation $z^2-2(m+a)z+m^2=0$ (never into $\Delta'$), producing a new quadratic equation in $m$; solve each for $m$ (each may give 0, 1, or 2 real values of $m$; if `solve(...)` returns an empty list, that branch simply contributes nothing, move on). For every $m$ found this way, check whether it satisfies $\Delta'>0$. If it passes, increment `count = count + 1`; if not, discard.
6. **Why no value of $m$ is ever double-counted.** Case 1 requires $\Delta'=0$, Case 2 requires $\Delta'<0$, Case 3 requires $\Delta'>0$; these three conditions on $m$ are mutually exclusive (a single value of $m$ satisfies exactly one of them, never two), so a value accepted in one case can never also be accepted in another; adding up the counts from all three cases directly, with no cross-checking between cases, is always correct.
7. **Match.** Once all three cases have been processed, `count` already holds the exact number of valid $m$. Test the four options in order with `Eq(count, <option>)` and stop at the first `True`; never pick the letter by eye.
'''


# =====================================================================
# [D] PART_FEWSHOT - VI DU MAU RIENG CUA DANG D47 (THAY KHI DOI DANG)
# Cac dong "Observation:" duoi day la KET QUA THAT (chay bang chinh class
# MayTinh o backend, khong phai sympy roi rac va khong phai so bia).
# =====================================================================
PART_FEWSHOT = r'''
**PART 3: WORKED EXAMPLE (STUDY THIS TRACE CAREFULLY)**

Trên tập hợp các số phức, xét phương trình $z^2-2(m+1)z+m^2=0$ ($m$ là tham số thực). Có bao nhiêu giá trị của $m$ để phương trình đó có nghiệm $z_0$ thỏa mãn $|z_0|=5$?
A. $2$ | B. $3$ | C. $1$ | D. $4$

<think>
Thought: Read off a and R. The equation is z^2-2(m+1)z+m^2=0, matching z^2-2(m+a)z+m^2=0 with a=1. The target modulus is R=5. I will now type exactly: delta_prime = expand((m+1)**2 - m**2)
Action: Calculator
Action Input: delta_prime = expand((m+1)**2 - m**2)
Observation: delta_prime = 2*m + 1

Thought: Start the running tally at 0 before working through the three cases.
Action: Calculator
Action Input: count = 0
Observation: count = 0

Thought: Case 1: delta_prime=0. Solve for m.
Action: Calculator
Action Input: roots_case1 = solve(Eq(delta_prime, 0), m)
Observation: roots_case1 = [-1/2]

Thought: Take that root.
Action: Calculator
Action Input: m1 = roots_case1[0]
Observation: m1 = -1/2

Thought: For this m the double root is z0=a+m=1+m1. I will now type exactly: z0_th1 = m1 + 1
Action: Calculator
Action Input: z0_th1 = m1 + 1
Observation: z0_th1 = 1/2

Thought: Check |z0|=R=5.
Action: Calculator
Action Input: Eq(Abs(z0_th1), 5)
Observation: False

Thought: Case 1 fails, discarded, count stays 0. Case 2: delta_prime<0. Write the root formula: z0=(a+m)+-sqrt(|delta_prime|)i. Here delta_prime<0 so |delta_prime|=-delta_prime, giving z0=(a+m)+-sqrt(-delta_prime)i. Square the modulus term by term: |z0|^2=(a+m)^2+(-delta_prime). Substitute delta_prime=(a+m)^2-m^2: |z0|^2=(a+m)^2-[(a+m)^2-m^2]=m^2. So |z0|=R iff m^2=R^2 iff m=R or m=-R, i.e. m=5 or m=-5. Check the first candidate m=5 against delta_prime<0.
Action: Calculator
Action Input: delta_prime.subs(m, 5) < 0
Observation: False

Thought: m=5 fails the Case 2 condition, discarded. Check the second candidate m=-5.
Action: Calculator
Action Input: delta_prime.subs(m, -5) < 0
Observation: True

Thought: m=-5 passed; increment the tally.
Action: Calculator
Action Input: count = count + 1
Observation: count = 1

Thought: Case 3: delta_prime>0. Substitute z0=R=5 directly into the ORIGINAL equation (not into delta_prime). I will now type exactly: eq_R = expand(5**2 - 2*(m+1)*5 + m**2)
Action: Calculator
Action Input: eq_R = expand(5**2 - 2*(m+1)*5 + m**2)
Observation: eq_R = m**2 - 10*m + 15

Thought: Solve it for m.
Action: Calculator
Action Input: roots_R = solve(Eq(eq_R, 0), m)
Observation: roots_R = [5 - sqrt(10), sqrt(10) + 5]

Thought: Take the first root and check it against delta_prime>0.
Action: Calculator
Action Input: m3a = roots_R[0]
Observation: m3a = 5 - sqrt(10)

Action: Calculator
Action Input: delta_prime.subs(m, m3a) > 0
Observation: True

Thought: Passed; increment the tally.
Action: Calculator
Action Input: count = count + 1
Observation: count = 2

Thought: Take the second root the same way.
Action: Calculator
Action Input: m3b = roots_R[1]
Observation: m3b = sqrt(10) + 5

Action: Calculator
Action Input: delta_prime.subs(m, m3b) > 0
Observation: True

Thought: Passed too; increment the tally.
Action: Calculator
Action Input: count = count + 1
Observation: count = 3

Thought: Now substitute z0=-R=-5 into the ORIGINAL equation. I will now type exactly: eq_negR = expand((-5)**2 - 2*(m+1)*(-5) + m**2)
Action: Calculator
Action Input: eq_negR = expand((-5)**2 - 2*(m+1)*(-5) + m**2)
Observation: eq_negR = m**2 + 10*m + 35

Thought: Solve it for m.
Action: Calculator
Action Input: roots_negR = solve(Eq(eq_negR, 0), m)
Observation: roots_negR = []

Thought: Empty list, this branch contributes nothing. All three cases are now processed; the running tally already holds the final count. Read option A verbatim from the problem, one option at a time: "A. $2$"; A is 2. Test it.
Action: Calculator
Action Input: Eq(count, 2)
Observation: False

Thought: Not A. Read option B verbatim: "B. $3$"; B is 3. Test it.
Action: Calculator
Action Input: Eq(count, 3)
Observation: True

Option B matches; stop here.
</think>
Final Answer: \boxed{B}
'''


# =====================================================================
# [E] PART_NHIEMVU - CHOT NHIEM VU (sua danh sach buoc khi doi dang)
# =====================================================================
PART_NHIEMVU = r'''
**PART 4: YOUR TURN**

Open a `<think>` tag as the very first thing you write, and close it with `</think>`. Think in English inside the tags. Work through the method of PART 2, and use a Calculator call for every computation; never compute anything yourself:

[Step 1] Match the given equation against $z^2-2(m+a)z+m^2=0$ and read off $a$ and $R$ in your Thought, going term by term and watching the sign of $a$.
[Step 2] Compute `delta_prime = expand((m+<a>)**2 - m**2)`, then set `count = 0`.
[Step 3] Case 1 ($\Delta'=0$): compute `roots_case1 = solve(Eq(delta_prime, 0), m)`; for each root, compute `z0 = <a> + <root>` and check `Eq(Abs(z0), <R>)`; if it passes, `count = count + 1`.
[Step 4] Case 2 ($\Delta'<0$): in your Thought, write out the root formula $z_0=(a+m)\pm\sqrt{|\Delta'|}\,i$, note $|\Delta'|=-\Delta'$ since $\Delta'<0$, square the modulus term by term, then substitute $\Delta'=(a+m)^2-m^2$ to arrive at $|z_0|^2=m^2$; do not skip straight to "the candidates are $m=\pm R$" without this derivation. This gives the two candidates $m=R$ and $m=-R$; for each one, check `delta_prime.subs(m, <candidate>) < 0`; if it passes, `count = count + 1`.
[Step 5] Case 3 ($\Delta'>0$): substitute $z=R$ and separately $z=-R$ directly into the ORIGINAL equation $z^2-2(m+a)z+m^2=0$ (never into `delta_prime`) to build two new quadratics in $m$; solve each with `solve(...)` (an empty list means that branch contributes nothing); for every root found, check `delta_prime.subs(m, <root>) > 0`; if it passes, `count = count + 1`.
[Step 6] The three cases' conditions on $m$ ($\Delta'=0$, $<0$, $>0$) are mutually exclusive, so no cross-checking between cases is needed; once all three are processed, `count` already holds the exact answer.
[Step 7] Go through the options ONE AT A TIME in order. For each one, quote its exact text from the problem verbatim in your Thought before assigning it a value, then test that value with `Eq(count, <that value>)`, and stop at the first `True`. Never pick the letter by eye.

Right before each Calculator call, write a Thought that spells out the EXACT line you are about to type (for example "I will now type exactly: eq_R = ..."), then copy that line character-for-character into `Action Input` rather than retyping it from the earlier prose.

Then, immediately after `</think>`, write exactly:
Final Answer: \boxed{<Letter>}

Đề bài:
{de_bai}
'''


PROMPT_TEMPLATE = (PART_TOOL + PART_KIENTHUC + PART_HUONGGIAI
                   + PART_FEWSHOT + PART_NHIEMVU)


# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN

In [ ]:
# BACKEND: TOOL "Calculator" - may tinh sympy CO BO NHO BIEN
# =====================================================================
# Tong quat cho MOI dang toan (khong chi D53): nhan 1 bieu thuc sympy bat
# ky, tra ve gia tri chinh xac tuyet doi. Ho tro:
#   - gan bien:  "ten = bieu_thuc"  -> luu vao bo nho, dung lai o luot sau
#     (xoa han lo hoi model chep tay lai so dai - nguon loi lon nhat)
#   - so khop :  "Eq(a, b)"         -> True/False chinh xac, khong xap xi
#   - moi phep cong tru nhan chia phan so, can thuc, so phuc, mo dun, lien hop
import sympy as sp

GAN_BIEN_RE = re.compile(r'^([A-Za-z_]\w*)\s*=\s*(.+)$', re.S)


class MayTinh:
    """May tinh sympy co bo nho bien (reset moi cau).

    an_so_tu_do: tap ten bien duoc PHEP giu tu do trong ket qua (khong bao
    loi 'undefined') du chua tung duoc gan gia tri - dung cho cac dang co
    an so chua biet can giai (vi du D35 dung x, y la an can tim, chi tro
    thanh so cu the sau buoc solve() o gan cuoi)."""

    def __init__(self, an_so_tu_do=()):
        self.an_so_tu_do = set(an_so_tu_do)
        # DANG KY SAN moi an so tu do la ky hieu SO THUC (real=True) ngay tu
        # dau - neu de sympify tu tao ky hieu (khi gap ten lan dau trong 1
        # bieu thuc), no se KHONG mac dinh la so thuc, khien Abs(t-4+I) khong
        # tu rut gon duoc thanh (t-4)**2+1 ma giu nguyen dang chua rut gon.
        self.ns = {ten: sp.Symbol(ten, real=True) for ten in self.an_so_tu_do}

    def _kiem_tra_ten_la(self, bieu_thuc):
        """sympify AM THAM bien ten chua dinh nghia thanh Symbol rong (khong
        bao loi) -> ket qua se vo nghia ma model khong he hay biet. Chan lai
        moi ten KHONG nam trong bo nho VA khong nam trong danh sach an so
        tu do duoc khai bao truoc."""
        con_lai = getattr(bieu_thuc, 'free_symbols', set())
        ten_thieu = {str(x) for x in con_lai} - self.an_so_tu_do
        if not ten_thieu:
            return None
        ten_la = ', '.join(sorted(ten_thieu))
        da_co = ', '.join(sorted(self.ns)) or '(chua co bien nao)'
        an_cho_phep = ', '.join(sorted(self.an_so_tu_do)) or '(khong co)'
        return (f'undefined name(s): {ten_la}. You used a name that is not in '
                f'calculator memory and is not a declared free unknown. Names '
                f'currently in memory: {da_co}. Declared free unknowns (allowed '
                f'without prior assignment): {an_cho_phep}. Either compute and '
                'store that name first (with "name = expression"), or rewrite '
                'the expression without it.')

    def tinh(self, bieu_thuc: str):
        """Tra ve (result_str, loi_str). Loi thi result_str = None."""
        s = bieu_thuc.strip()
        if not s:
            return None, 'empty expression'
        if '==' in s:
            return None, ('"==" is not supported. To compare two expressions '
                          'exactly, write Eq(left, right) instead.')

        ten = None
        m = GAN_BIEN_RE.match(s)
        # "Eq(a, b)" cung khop regex tren neu viet la "x = Eq(...)"; con
        # "Eq(a,b)" thuan thi khong co dau "=" o cap ngoai nen khong khop.
        if m and not s.lstrip().startswith('Eq('):
            ten, s = m.group(1), m.group(2)

        try:
            parsed = sp.sympify(s, locals=self.ns)
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'

        try:
            if isinstance(parsed, sp.Equality):
                loi_ten = self._kiem_tra_ten_la(parsed.lhs - parsed.rhs)
                if loi_ten:
                    return None, loi_ten
                bang_nhau = sp.simplify(parsed.lhs - parsed.rhs) == 0
                return ('True' if bang_nhau else 'False'), None

            if isinstance(parsed, (list, tuple, sp.FiniteSet)):
                # Ket qua tra ve tu solve(...): danh sach nghiem. Khong goi
                # expand/simplify tren list - kiem tra tung phan tu rieng.
                ds = list(parsed)
                for phan_tu in ds:
                    loi_ten = self._kiem_tra_ten_la(phan_tu)
                    if loi_ten:
                        return None, loi_ten
                if ten:
                    self.ns[ten] = ds
                    return f'{ten} = {ds}', None
                return str(ds), None

            gia_tri = sp.expand(sp.simplify(parsed))
            loi_ten = self._kiem_tra_ten_la(gia_tri)
            if loi_ten:
                return None, loi_ten
            if ten:
                self.ns[ten] = gia_tri
                return f'{ten} = {gia_tri}', None
            return str(gia_tri), None
        except Exception as e:
            return None, f'{type(e).__name__}: {e}'


# =====================================================================
# VONG LAP ReAct THEO LO (batch) - khac biet duy nhat so voi ban 1 cau
# =====================================================================
STOP_STR = 'Observation:'
MAX_TOOL_CALLS = 35          # van an toan chong loop vo han (D47: toi da ~20 luot tinh 3 truong hop + 4 luot doi chieu phuong an)
ACTION_INPUT_RE = re.compile(r'Action Input:[ \t]*(.*)')


class TrangThai:
    """Trang thai ReAct rieng cua 1 cau (bo nho bien rieng, ngan sach rieng)."""

    def __init__(self, rec, chat_prompt):
        self.rec = rec
        self.chat_prompt = chat_prompt
        self.may_tinh = MayTinh(an_so_tu_do=AN_SO_TU_DO)   # bo nho + an so RIENG cho tung cau
        # EP SAN token dau tien (giong ban 1 cau): model bi buoc noi tiep tu
        # "Thought:" ngay sau <think>, khong con quyen tu chon viet van xuoi
        # mo dau (hanh vi mac dinh de lech khoi dinh dang ReAct).
        self.full_text = '<think>\nThought:'
        self.tong_tok = 0
        self.n_calls = 0
        self.nhat_ky = []
        self.xong = False
        self.ly_do_dung = ''
        self.buoc_chot = False             # het luot tool -> ep viet Final Answer


prompts_ban_dau = []
for r in records:
    prompt_text = PROMPT_TEMPLATE.replace('{de_bai}', r['de_bai_mcq'])
    prompts_ban_dau.append(tok.apply_chat_template(
        [{'role': 'user', 'content': prompt_text}],
        tokenize=False, add_generation_prompt=True, enable_thinking=True))

ds = [TrangThai(r, p) for r, p in zip(records, prompts_ban_dau)]

t0 = time.time()
vong = 0
while True:
    hoat_dong = [s for s in ds if not s.xong]
    if not hoat_dong:
        break
    vong += 1

    lo = []
    for s in hoat_dong:
        dau_vao = s.chat_prompt + s.full_text
        cho_trong = MAX_MODEL_LEN - len(tok(dau_vao).input_ids) - 8
        so_sinh = min(MAX_NEW_TOKENS - s.tong_tok, cho_trong)
        if so_sinh <= 0:
            s.xong = True
            s.ly_do_dung = 'het_cho_context' if cho_trong <= 0 else 'het_ngan_sach_token'
            continue
        stop = None if s.buoc_chot else [STOP_STR]
        lo.append((s, dau_vao, SamplingParams(
            temperature=TEMPERATURE, top_p=TOP_P, top_k=TOP_K,
            presence_penalty=PRESENCE_PENALTY, max_tokens=so_sinh,
            seed=SEED, stop=stop)))

    if not lo:
        break

    outs = llm.generate([p for _, p, _ in lo],
                        [sp for _, _, sp in lo], use_tqdm=False)

    n_goi_vong_nay = 0
    for (s, _, _), out in zip(lo, outs):
        o = out.outputs[0]
        s.full_text += o.text
        s.tong_tok += len(o.token_ids)

        if s.buoc_chot:
            s.xong = True
            s.ly_do_dung = 'het_han_muc_tool'
            continue

        if o.stop_reason != STOP_STR:
            s.xong = True
            s.ly_do_dung = ('model_ket_thuc' if o.finish_reason == 'stop'
                            else 'het_token')
            continue

        s.n_calls += 1
        n_goi_vong_nay += 1
        khop = None
        for mm in ACTION_INPUT_RE.finditer(o.text):
            khop = mm
        bieu_thuc = khop.group(1).strip() if khop else ''

        if not bieu_thuc:
            quan_sat = ('ERROR: no "Action Input:" line found. Write a Thought, '
                        'then "Action: Calculator", then "Action Input: <expression>".')
            s.nhat_ky.append(('(khong co Action Input)', quan_sat))
        else:
            ket_qua, loi = s.may_tinh.tinh(bieu_thuc)
            if loi:
                quan_sat = (f'ERROR: {loi}. Fix the syntax and try again '
                            '(use I for the imaginary unit, sqrt() for roots, '
                            '** for powers, Abs()/conjugate(), Eq(a,b) to compare).')
                s.nhat_ky.append((bieu_thuc, f'ERROR: {loi}'))
            else:
                quan_sat = ket_qua
                s.nhat_ky.append((bieu_thuc, ket_qua))

        s.full_text += f'{STOP_STR} {quan_sat}\n'

        if s.n_calls >= MAX_TOOL_CALLS:
            s.full_text += ('\n[SYSTEM: tool-call limit reached - give your Final '
                            'Answer now, with no further Action.]\n')
            s.buoc_chot = True

    print(f'  Vong {vong:2d}: {len(lo):2d} cau sinh, {n_goi_vong_nay:2d} luot goi tool, '
          f'con lai {sum(1 for s in ds if not s.xong):2d} cau '
          f'({time.time()-t0:.0f}s)')

print(f'Xong toan bo sau {vong} vong, {(time.time()-t0)/60:.1f} phut.')


# =====================================================================
# CHAM DIEM + LUU KET QUA
# =====================================================================
def lay_dap_an(t):
    for pat in (r'Final\s+Answer[^A-D]{0,20}([A-D])\b',
                r'\\boxed\{\s*([A-D])\s*\}',
                r'[Đđ]áp\s*án[^A-D]{0,15}([A-D])\b'):
        m = re.findall(pat, t)
        if m:
            return m[-1]
    return ''


rows = []
for s in ds:
    chon = lay_dap_an(s.full_text)
    n_loi = sum(1 for _, obs in s.nhat_ky if str(obs).startswith('ERROR'))
    rows.append({
        'STT': s.rec['STT'],
        'loai_so': s.rec.get('loai_so', ''),
        'dap_an_dung': s.rec['dap_an_letter'],
        'model_chon': chon,
        'DUNG': chon == s.rec['dap_an_letter'],
        'token': s.tong_tok,
        'so_luot_goi_tool': s.n_calls,
        'so_loi_cu_phap': n_loi,
        'ly_do_dung': s.ly_do_dung,
        'nhat_ky_tool': ' | '.join(f'{bt} -> {obs}' for bt, obs in s.nhat_ky),
        'full_text': s.full_text,
    })

df = pd.DataFrame(rows)
df.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')

print('=' * 72)
print(f"KET QUA: {df['DUNG'].sum()} / {len(df)} DUNG ({df['DUNG'].mean()*100:.1f}%)")
print('=' * 72)
print()
print('--- Theo loai so ---')
print(df.groupby('loai_so')['DUNG'].agg(['sum', 'count', 'mean']))
print()
print('--- Chat luong dung tool ---')
print(f"Trung binh luot goi tool/cau : {df['so_luot_goi_tool'].mean():.1f}")
print(f"Tong luot goi tool           : {df['so_luot_goi_tool'].sum()}")
print(f"Tong luot LOI cu phap        : {df['so_loi_cu_phap'].sum()} "
      f"({df['so_loi_cu_phap'].sum() / max(1, df['so_luot_goi_tool'].sum()) * 100:.1f}%)")
print(f"So cau KHONG goi tool lan nao: {(df['so_luot_goi_tool'] == 0).sum()}")
print(f"Trung binh token/cau         : {df['token'].mean():.0f}")
print()
print('--- Ly do dung ---')
print(df['ly_do_dung'].value_counts())
print()
if (~df['DUNG']).any():
    print('--- CAC CAU SAI ---')
    print(df[~df['DUNG']][['STT', 'loai_so', 'dap_an_dung', 'model_chon',
                           'token', 'so_luot_goi_tool', 'so_loi_cu_phap',
                           'ly_do_dung']].to_string(index=False))
else:
    print(f"*** TAT CA {len(df)} CAU DEU DUNG ***")
print()
print('Da luu:', OUT_PATH)